# Exercise 02, Part 4 — Flagging bad GNSS signals with a classifier

Run this in **Google Colab**, signed in with your WashU Google account. Nothing
installs on your own computer.

**What you are doing.** A GNSS receiver assumes each satellite signal travelled
in a straight line to the antenna. Near buildings and under tree canopy a signal
can bounce off a surface first, arriving late and making the satellite look
farther away than it is. A position built from bounced signals is pulled off,
often by metres. This model reads your RINEX file and, for every satellite at
every epoch, predicts whether that signal arrived cleanly (**LOS**,
line-of-sight) or bounced (**NLOS**, non-line-of-sight).

It does not fix your position. It flags signals.

**What is graded.** Running the model is mechanical and takes one cell. Your
grade is on Part 4.3 below: judging whether the output makes sense for the site
you actually stood on. A model that flags almost nothing at an open-sky site may
well be right, and saying so with evidence is a full-credit answer.

---
### How to run a Colab notebook
Click a cell, then press **Shift+Enter** to run it and move to the next one. Run
the cells in order from the top. A spinning circle at the left means the cell is
still working; wait for it to finish before running the next.

## Step 1 — Install the one package Colab does not already have

This takes about 20 seconds. `pandas`, `numpy`, `scikit-learn` and `matplotlib` are already in Colab.

In [ ]:
!pip install --quiet pyrtklib
import pyrtklib, sklearn, sys
print("python     ", sys.version.split()[0])
print("pyrtklib   ", "ok")
print("scikit-learn", sklearn.__version__)

## Step 2 — Fetch the feature extractor and the training data

Two files come from the course repository. `extract_features.py` turns a RINEX
file into the five numbers the model uses, and it is the same code the course
container runs, so features are computed identically either way.
`nlos_features.csv.gz` is the labelled training table: 26,522 satellite
observations from Hong Kong with a known LOS/NLOS answer for each.

We train the model here rather than shipping a pre-trained one. A saved
scikit-learn model only loads reliably under the scikit-learn version that wrote
it, and Colab's version changes without warning. Training takes a second.

In [ ]:
import urllib.request, pathlib

BASE = "https://raw.githubusercontent.com/fossettlab/gnss-nlos-notebook/main"
for name, dest in [
    ("extract_features.py", "extract_features.py"),
    ("nlos_features.csv.gz",       "nlos_features.csv.gz"),
]:
    urllib.request.urlretrieve(f"{BASE}/{name}", dest)
    print(f"{dest:26s} {pathlib.Path(dest).stat().st_size/1e6:7.2f} MB")

## Step 2b — Train the classifier, and see how well it does on data it has not seen

The training data covers four separate collection runs. To get an honest score
we hold out one entire run, train on the other three, and test on the held-out
one, repeating for each run. This is **leave-one-run-out**.

It matters that we hold out a whole run rather than shuffling rows: observations
seconds apart are nearly identical, so a random split would let the model train
on data adjacent to its own test set and report a flattering score.

In [ ]:
import time, pandas as pd, numpy as np
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

FEATURES = ["cn0", "elev_deg", "pr_resid_norm", "pr_rate_consistency", "cn0_std"]
train = pd.read_csv("nlos_features.csv.gz")
print(f"{len(train):,} labelled observations, {train['nlos'].mean():.1%} of them NLOS, "
      f"{train['run'].nunique()} runs")

def make_model():
    return Pipeline([
        ("impute", SimpleImputer(strategy="median")),
        ("scale",  StandardScaler()),
        ("clf",    RandomForestClassifier(n_estimators=300, min_samples_leaf=5,
                                          random_state=42, n_jobs=-1)),
    ])

t0 = time.perf_counter()
scores = []
for held_out in sorted(train["run"].unique()):
    te = train[train["run"] == held_out]
    tr = train[train["run"] != held_out]
    m = make_model().fit(tr[FEATURES], tr["nlos"])
    prob = m.predict_proba(te[FEATURES])[:, 1]
    pred = (prob >= 0.5).astype(int)
    scores.append({"held out": held_out, "n": len(te),
                   "accuracy": accuracy_score(te["nlos"], pred),
                   "F1": f1_score(te["nlos"], pred),
                   "ROC-AUC": roc_auc_score(te["nlos"], prob)})
print(pd.DataFrame(scores).round(3).to_string(index=False))
print()
print(f"mean accuracy {np.mean([s['accuracy'] for s in scores]):.3f}   "
      f"mean F1 {np.mean([s['F1'] for s in scores]):.3f}")

model = make_model().fit(train[FEATURES], train["nlos"])
print()
print(f"final model trained on all {len(train):,} rows in {time.perf_counter()-t0:.1f}s total")

**Read those numbers before going further.** An F1 near 0.76 means roughly one in
four real NLOS signals is missed and roughly one in five NLOS calls is actually a
fine signal. That is the accuracy you are about to apply to your own data.

A random forest also reports which features it leaned on:

In [ ]:
imp = pd.Series(model.named_steps["clf"].feature_importances_, index=FEATURES)
print(imp.sort_values(ascending=False).round(3).to_string())

## Step 3 — Upload your RINEX files

You need two files from your occupation:

- the **observation** file, ending `.YYo` or `.YYO` (`YY` is the year, so `.23O`)
- a **navigation** file, ending `.YYp`/`.YYP` or `.YYn`/`.YYN`

Run the cell, then pick both files in the dialog. A 2-hour observation file is
roughly 90 MB and the upload is the slow part; give it a few minutes.

*Prefer a shorter file for your first run.* Get the whole thing working on a
small file, then rerun on the long one.

In [ ]:
from google.colab import files
uploaded = files.upload()
for k, v in uploaded.items():
    print(f"{k:44s} {len(v)/1e6:8.1f} MB")

## Step 4 — Set your two filenames

Copy the names exactly as they appear above, including the extension.

In [ ]:
OBS_FILE = "rudolph_static_02hr.23O"      # <-- your observation file
NAV_FILE = "emlid-WU1-R_raw_20230913170733.23P"   # <-- your navigation file

import pathlib
for f in (OBS_FILE, NAV_FILE):
    p = pathlib.Path(f)
    print(("found   " if p.exists() else "MISSING "), f)

## Step 5 — Extract the features and classify

For every satellite at every epoch this computes five numbers, then asks the
model for a verdict:

| Feature | What it measures |
|---|---|
| `cn0` | signal strength, dB-Hz. Bounced signals lose energy |
| `elev_deg` | satellite's angle above the horizon. Low signals skim obstructions |
| `pr_resid_norm` | how far the measured range sits from what a position solve expects |
| `pr_rate_consistency` | whether Doppler agrees with how fast the range is changing |
| `cn0_std` | how much signal strength wobbles over five epochs |

Expect roughly **half a minute for a 2-hour file** and a couple of minutes for
an 8-hour one.

In [ ]:
import sys
sys.path.insert(0, ".")
from extract_features import read_rinex, make_prcopt, raw_epoch_rows, add_derived_features

t0 = time.perf_counter()
obs, nav = read_rinex(pathlib.Path(OBS_FILE), [pathlib.Path(NAV_FILE)])
rows = raw_epoch_rows(obs, nav, make_prcopt())
df = add_derived_features(pd.DataFrame(rows))
print(f"extracted {len(df):,} (epoch, satellite) rows in {time.perf_counter()-t0:.1f} s")

df["nlos_prob"] = model.predict_proba(df[FEATURES])[:, 1]
df["nlos_pred"] = (df["nlos_prob"] >= 0.5).astype(int)
df.to_csv("nlos_predictions.csv", index=False)

flagged = df["nlos_pred"].mean()
print(f"flagged NLOS: {df['nlos_pred'].sum():,} of {len(df):,} rows ({flagged:.1%})")
df.head()

## Part 4.2 — Plot the probability against satellite elevation

The reasoning behind the model is that low, weak, jumpy signals are the likely
bounces. This plot is where you check whether it behaves that way on your data.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7, 4.5))
for pred, label, colour in [(0, "predicted LOS", "#2E4756"), (1, "predicted NLOS", "#B4451F")]:
    sub = df[df["nlos_pred"] == pred]
    ax.scatter(sub["elev_deg"], sub["nlos_prob"], s=6, alpha=0.3,
               c=colour, label=f"{label} (n={len(sub):,})")
ax.axhline(0.5, color="grey", lw=0.8, ls="--")
ax.set_xlabel("satellite elevation above horizon (degrees)")
ax.set_ylabel("model probability the signal is NLOS")
ax.set_title("NLOS probability vs elevation")
ax.legend(markerscale=3, frameon=False)
fig.tight_layout()
fig.savefig("nlos_vs_elevation.png", dpi=150)
plt.show()

**Write about what you see.** Does the flagged group sit at lower elevations?
Does it have weaker `cn0`? If the pattern is absent, say that: it is a finding,
not a failure to produce one.

A second view that often helps:

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
for pred, label, colour in [(0, "predicted LOS", "#2E4756"), (1, "predicted NLOS", "#B4451F")]:
    sub = df[df["nlos_pred"] == pred]
    ax.scatter(sub["elev_deg"], sub["cn0"], s=6, alpha=0.3, c=colour, label=label)
ax.set_xlabel("satellite elevation above horizon (degrees)")
ax.set_ylabel("C/N0, signal strength (dB-Hz)")
ax.set_title("Signal strength vs elevation, coloured by the model's call")
ax.legend(markerscale=3, frameon=False)
fig.tight_layout()
plt.show()

print(df.groupby("nlos_pred")[list(FEATURES)].mean().round(2).to_string())

## Part 4.3 — Evaluate the model on your own data (this is the graded part)

The model was trained on **KLTDataset**: a GNSS collection from Kowloon Tong,
Hong Kong, a dense high-rise urban canyon. Ground truth came from a sky-pointing
fisheye camera, so satellites projecting onto open sky were labelled LOS and ones
projecting onto a building were labelled NLOS. 26,522 labelled observations from
four runs on one day, 36% of them NLOS.

You measured its held-out performance yourself in Step 2b. For reference, the
random forest scores roughly **0.83 accuracy, 0.75 F1, and ROC-AUC between 0.88
and 0.91** averaged across held-out runs. Your numbers will differ in the third
decimal place depending on the scikit-learn version Colab is running that week. An RBF support-vector machine on the same data
scores within a thousandth of that on F1 while taking twenty-five times longer to
train, which is why this notebook uses the forest.

**Whether any of that transfers to a St. Louis field site is untested.**

Answer these in your report:

1. **What fraction of your rows did it flag, and does that match your sky?**
   You were there. Was there dense canopy, a building edge, open field? If the
   model flagged 2% at an open-sky site, that is plausibly correct.
2. **Which error would hurt your survey more** — discarding a good signal, or
   trusting a bounced one? Justify it from what you were measuring.
3. **Do the flags line up with anything physical?** Use the group means printed
   above. Are flagged signals actually weaker and lower?
4. **Look at `nlos_prob`, not just the flag.** How many of your rows sit between
   0.4 and 0.6, where the model is effectively undecided?
5. **The training labels are themselves a heuristic.** A fisheye camera's
   sky-versus-building call is wrong near edges, so 83% accuracy is measured
   against an imperfect reference. What does that do to how much you trust it?

In [ ]:
undecided = df["nlos_prob"].between(0.4, 0.6).mean()
print(f"rows the model is effectively undecided about (0.4 to 0.6): {undecided:.1%}")
print(f"confidently NLOS (>0.9): {(df['nlos_prob'] > 0.9).mean():.1%}")
print(f"confidently LOS  (<0.1): {(df['nlos_prob'] < 0.1).mean():.1%}")

## Part 4.4 (optional) — Does excluding the flagged signals move your position?

Recompute your Part 2 or Part 3 position with the flagged epochs and satellites
dropped, and compare. Before you look: which direction would you expect it to
move, and why?

## Step 6 — Download your results

Take `nlos_predictions.csv` and the figures into your report.

In [ ]:
from google.colab import files
files.download("nlos_predictions.csv")
files.download("nlos_vs_elevation.png")

---
### Citation

Hu, R., Wen, W., & Hsu, L.-T. (2023). Fisheye camera aided NLOS exclusion and
learning-based pseudorange correction. *2023 IEEE 26th International Conference
on Intelligent Transportation Systems (ITSC)*, 6088–6095.